# H2O Denoiser And Diffusion Debug

这个 notebook 使用单帧 `h2o_react.extxyz` 和单帧 `h2o_product.extxyz`，检查：

`RxnDataset -> Diffusion.forward -> Denoiser.forward -> backward`

重点是 shape、dtype、NaN、Denoiser 等变性、Diffusion 加噪、loss forward 和 backward。

In [1]:
from pathlib import Path
import sys

import torch
from e3nn import o3
from torch.utils.data import DataLoader

repo_root = Path.cwd()
if not (repo_root / 'dataset').exists() and (repo_root.parent / 'dataset').exists():
    repo_root = repo_root.parent
print(repo_root)

if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

from akmcgc.dataset import RxnDataset
from akmcgc.denoising import Denoiser
from akmcgc.diffusion import DiffSchedule, Diffusion, Norm, PredefinedNoiseSchedule
from akmcgc.model import EGNN

torch.manual_seed(2024)
data_dir = repo_root / 'tests' / 'data'


/Users/wx/Desktop/yyxwjq/akmcgc


## 0. Debug Contract Helpers

这些 helper 会在每一层边界打印：输入 tensor 维度、输出 tensor 维度、模块参数矩阵维度。


In [2]:
def tensor_contract(name, value):
    if torch.is_tensor(value):
        return f"{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}"
    return f"{name:16s} value={value!r}"


def print_tensor_contract(title, tensors):
    print(f"\n[{title}]")
    for name, value in tensors.items():
        print(tensor_contract(name, value))


def print_parameter_contract(title, module, max_rows=30):
    print(f"\n[{title} parameter contract]")
    total = 0
    trainable = 0
    rows = []
    for name, param in module.named_parameters():
        n = param.numel()
        total += n
        if param.requires_grad:
            trainable += n
        rows.append((name, tuple(param.shape), param.dtype, param.requires_grad, n))
    print(f"total parameters={total:,}, trainable={trainable:,}, tensors={len(rows)}")
    for name, shape, dtype, requires_grad, n in rows[:max_rows]:
        print(f"{name:48s} shape={shape!s:18s} dtype={str(dtype):14s} trainable={requires_grad!s:5s} n={n:,}")
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more parameter tensors")


## 1. Load One H2O Reactant/Product Pair

In [3]:
dataset = RxnDataset(
    react_file=str(data_dir / 'h2o_react.extxyz'),
    product_file=str(data_dir / 'h2o_product.extxyz'),
    cutoff=6.0,
    max_neigh=16,
    r_fixed=True,
    r_pbc=True,
    device='cpu',
)
loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=RxnDataset.collate_fn)
batch = next(iter(loader))

for key in ['h', 'pos', 'edge_index', 'cell_offsets', 'cell', 'pbc', 'fragment', 'mask', 'neighbors']:
    value = batch[key]
    print(f'{key:12s}', tuple(value.shape), value.dtype)

assert len(dataset) == 1
assert batch['h'].dtype == torch.float64
assert torch.allclose(batch['h'][:, :3], batch['pos'])


h            (6, 122) torch.float64
pos          (6, 3) torch.float64
edge_index   (2, 12) torch.int64
cell_offsets (12, 3) torch.float64
cell         (1, 2, 3, 3) torch.float64
pbc          (1, 2, 3) torch.bool
fragment     (6,) torch.int64
mask         (6,) torch.int64
neighbors    (1,) torch.int64


## 1.1 Dataset Output Contract

这就是 dataset / collate 交给后续 Denoiser 和 Diffusion 的完整 batch contract。


In [4]:
print_tensor_contract('RxnDataset.collate_fn output batch', {
    'h=[pos,one_hot,charge]': batch['h'],
    'pos': batch['pos'],
    'edge_index': batch['edge_index'],
    'cell_offsets': batch['cell_offsets'],
    'cell': batch['cell'],
    'pbc': batch['pbc'],
    'fragment': batch['fragment'],
    'mask': batch['mask'],
    'neighbors': batch['neighbors'],
    'n_is': batch['n_is'],
    'n_fs': batch['n_fs'],
})



[RxnDataset.collate_fn output batch]
h=[pos,one_hot,charge] shape=(6, 122)           dtype=torch.float64  device=cpu
pos              shape=(6, 3)             dtype=torch.float64  device=cpu
edge_index       shape=(2, 12)            dtype=torch.int64    device=cpu
cell_offsets     shape=(12, 3)            dtype=torch.float64  device=cpu
cell             shape=(1, 2, 3, 3)       dtype=torch.float64  device=cpu
pbc              shape=(1, 2, 3)          dtype=torch.bool     device=cpu
fragment         shape=(6,)               dtype=torch.int64    device=cpu
mask             shape=(6,)               dtype=torch.int64    device=cpu
neighbors        shape=(1,)               dtype=torch.int64    device=cpu
n_is             shape=(1,)               dtype=torch.int64    device=cpu
n_fs             shape=(1,)               dtype=torch.int64    device=cpu


## 2. Build Denoiser

In [5]:
node_nf = batch['h'].shape[1]
model_input_dim = 32
# egnn图神经网络的配置参数
egnn_config = dict(
    in_node_nf=model_input_dim,
    in_edge_nf=0,
    hidden_nf=64,
    edge_hidden_nf=64,
    n_layers=2,
    attention=True,
    out_node_nf=model_input_dim,
    tanh=True,
    coords_range=10.0,
    norm_constant=1.0,
    inv_sublayers=2,
    sin_embedding=False,
    normalization_factor=1.0,
    aggregation_method='mean',
    reflect_equiv=True,
)
# 去噪器的配置参数
denoiser = Denoiser(
    model_config=egnn_config,
    fragment_names=['IS', 'FS'],
    node_nfs=[node_nf],
    edge_nf=0,
    condition_nf=0,
    pos_dim=3,
    update_pocket_coords=True,
    condition_time=True,
    model=EGNN,
    dtype=batch['h'].dtype,
).eval()
print('first parameter dtype:', next(denoiser.parameters()).dtype)
assert next(denoiser.parameters()).dtype == batch['h'].dtype


first parameter dtype: torch.float64


## 2.1 Denoiser Internal Contract

Denoiser 包装了三类参数：`node_encoder`、内部等变模型 `model`、`node_decoder`。下面打印每一类的输入/输出维度和参数矩阵维度。


In [6]:
print('\n[Denoiser scalar configuration]')
print('node_nf          =', denoiser.node_nf, '  # h = [pos, one_hot, charge]')
print('pos_dim          =', denoiser.pos_dim)
print('feature_dim      =', denoiser.node_nf - denoiser.pos_dim)
print('model_input_dim  =', denoiser.model_input_dim)
print('embed_dim        =', denoiser.embed_dim)
print('condition_time   =', denoiser.condition_time)
print('edge_nf          =', denoiser.edge_nf)
print('edge_encoder     =', denoiser.edge_encoder)
print('edge_decoder     =', denoiser.edge_decoder)

print_parameter_contract('Denoiser full module', denoiser, max_rows=36)
print_parameter_contract('Denoiser.node_encoder', denoiser.node_encoder, max_rows=10)
print_parameter_contract('Denoiser.model (EGNN)', denoiser.model, max_rows=24)
print_parameter_contract('Denoiser.node_decoder', denoiser.node_decoder, max_rows=10)



[Denoiser scalar configuration]
node_nf          = 122   # h = [pos, one_hot, charge]
pos_dim          = 3
feature_dim      = 119
model_input_dim  = 32
embed_dim        = 31
condition_time   = True
edge_nf          = 0
edge_encoder     = None
edge_decoder     = None

[Denoiser full module parameter contract]
total parameters=227,774, trainable=227,774, tensors=76
model.embedding.weight                           shape=(64, 32)           dtype=torch.float64  trainable=True  n=2,048
model.embedding.bias                             shape=(64,)              dtype=torch.float64  trainable=True  n=64
model.embedding_out.weight                       shape=(32, 64)           dtype=torch.float64  trainable=True  n=2,048
model.embedding_out.bias                         shape=(32,)              dtype=torch.float64  trainable=True  n=32
model.edge_embedding.weight                      shape=(63, 1)            dtype=torch.float64  trainable=True  n=63
model.edge_embedding.bias                      

## 3. Denoiser Forward: Shape / Dtype / NaN

In [7]:
num_graphs = int(batch['mask'].max().item()) + 1
t = torch.full((num_graphs, 1), 0.25, dtype=batch['h'].dtype, device=batch['h'].device)


## 3.1 Denoiser Forward Input Contract

Denoiser 的真实输入是已经处于当前 diffusion 时间步的 `h`，以及图结构和周期信息。这里先显式打印输入。


In [8]:
print_tensor_contract('Denoiser.forward inputs', {
    'h': batch['h'],
    'edge_index': batch['edge_index'],
    't': t,
    'mask': batch['mask'],
    'fragment': batch['fragment'],
    'cell': batch['cell'],
    'pbc': batch['pbc'],
    'cell_offsets': batch['cell_offsets'],
    'neighbors': batch['neighbors'],
})



[Denoiser.forward inputs]
h                shape=(6, 122)           dtype=torch.float64  device=cpu
edge_index       shape=(2, 12)            dtype=torch.int64    device=cpu
t                shape=(1, 1)             dtype=torch.float64  device=cpu
mask             shape=(6,)               dtype=torch.int64    device=cpu
fragment         shape=(6,)               dtype=torch.int64    device=cpu
cell             shape=(1, 2, 3, 3)       dtype=torch.float64  device=cpu
pbc              shape=(1, 2, 3)          dtype=torch.bool     device=cpu
cell_offsets     shape=(12, 3)            dtype=torch.float64  device=cpu
neighbors        shape=(1,)               dtype=torch.int64    device=cpu


In [9]:
with torch.no_grad():
    eps_h, edge_attr_out = denoiser(
        h=batch['h'],
        edge_index=batch['edge_index'],
        t=t,
        mask=batch['mask'],
        fragment=batch['fragment'],
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        conditions=None,
        edge_attr=None,
    )

print('eps_h', tuple(eps_h.shape), eps_h.dtype)
print('edge_attr_out', edge_attr_out)
print('max |eps_pos| ', float(eps_h[:, :3].abs().max()))
print('max |eps_feat|', float(eps_h[:, 3:].abs().max()))

assert eps_h.shape == batch['h'].shape
assert eps_h.dtype == batch['h'].dtype
assert torch.isfinite(eps_h).all()


eps_h (6, 122) torch.float64
edge_attr_out None
max |eps_pos|  0.26394398835759486
max |eps_feat| 0.2797499143559172


## 3.2 Denoiser Forward Output Contract

Denoiser 输出的是预测噪声 `eps_h`，shape 必须和输入 `h` 完全一致。


In [10]:
print_tensor_contract('Denoiser.forward outputs', {
    'eps_h=[eps_pos,eps_feat]': eps_h,
    'eps_pos': eps_h[:, :3],
    'eps_feat': eps_h[:, 3:],
    'edge_attr_out': edge_attr_out,
})



[Denoiser.forward outputs]
eps_h=[eps_pos,eps_feat] shape=(6, 122)           dtype=torch.float64  device=cpu
eps_pos          shape=(6, 3)             dtype=torch.float64  device=cpu
eps_feat         shape=(6, 119)           dtype=torch.float64  device=cpu
edge_attr_out    value=None


## 4. Denoiser Forward Equivariance

In [ ]:
def clone_batch(batch):
    return {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}


torch.manual_seed(0)
rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)

batch_rot = clone_batch(batch)
batch_rot['pos'] = batch['pos'] @ rot
batch_rot['cell'] = batch['cell'] @ rot
batch_rot['h'][:, :3] = batch_rot['pos']

with torch.no_grad():
    eps_h, _ = denoiser(
        h=batch['h'], edge_index=batch['edge_index'], t=t, mask=batch['mask'],
        fragment=batch['fragment'], cell=batch['cell'], pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'], neighbors=batch['neighbors'],
    )
    eps_h_rot, _ = denoiser(
        h=batch_rot['h'], edge_index=batch_rot['edge_index'], t=t, mask=batch_rot['mask'],
        fragment=batch_rot['fragment'], cell=batch_rot['cell'], pbc=batch_rot['pbc'],
        cell_offsets=batch_rot['cell_offsets'], neighbors=batch_rot['neighbors'],
    )

pos_err = float((eps_h[:, :3] @ rot - eps_h_rot[:, :3]).abs().max())
feat_err = float((eps_h[:, 3:] - eps_h_rot[:, 3:]).abs().max())
print('Denoiser max |eps_pos @ R - eps_pos_rot| =', pos_err)
print('Denoiser max |eps_feat - eps_feat_rot|    =', feat_err)

assert pos_err < 2e-5
assert feat_err < 2e-5


Denoiser max |eps_pos @ R - eps_pos_rot| = 3.397419234829613e-09
Denoiser max |eps_feat - eps_feat_rot|    = 3.929452763173913e-09


## 5. Build Diffusion And Check Noising

In [12]:
normalizer = Norm(norm_values=(1.0, 1.0, 1.0), norm_biases=(0.0, 0.0, 0.0), pos_dim=3)
gamma_module = PredefinedNoiseSchedule(noise_schedule='cosine', timesteps=32, precision=1e-5)
schedule = DiffSchedule(gamma_module=gamma_module, norm_values=(1.0, 1.0, 1.0))
ddpm = Diffusion(
    denoiser=denoiser,
    schdule=schedule,
    normalizer=normalizer,
    loss_type='l2',
    pos_only=True,
)



## 5.1 Diffusion Object Contract

Diffusion 本身负责加噪、时间调度、loss 和采样流程；它的可训练参数来自内部 `denoiser`。


In [13]:
print('\n[Diffusion scalar configuration]')
print('T              =', ddpm.T)
print('pos_dim        =', ddpm.pos_dim)
print('node_nf        =', ddpm.node_nf)
print('loss_type      =', ddpm.loss_type)
print('pos_only       =', ddpm.pos_only)
print('denoiser class =', type(ddpm.denoiser).__name__)
print_parameter_contract('Diffusion/ddpm', ddpm, max_rows=28)



[Diffusion scalar configuration]
T              = 32
pos_dim        = 3
node_nf        = 122
loss_type      = l2
pos_only       = True
denoiser class = Denoiser

[Diffusion/ddpm parameter contract]
total parameters=227,807, trainable=227,774, tensors=77
denoiser.model.embedding.weight                  shape=(64, 32)           dtype=torch.float64  trainable=True  n=2,048
denoiser.model.embedding.bias                    shape=(64,)              dtype=torch.float64  trainable=True  n=64
denoiser.model.embedding_out.weight              shape=(32, 64)           dtype=torch.float64  trainable=True  n=2,048
denoiser.model.embedding_out.bias                shape=(32,)              dtype=torch.float64  trainable=True  n=32
denoiser.model.edge_embedding.weight             shape=(63, 1)            dtype=torch.float64  trainable=True  n=63
denoiser.model.edge_embedding.bias               shape=(63,)              dtype=torch.float64  trainable=True  n=63
denoiser.model.edge_embedding_out.weight   

In [14]:
batch_norm = ddpm.normalizer.normalize_batch(batch)
t_noise = torch.full((num_graphs, 1), 0.5, dtype=batch['h'].dtype, device=batch['h'].device)
gamma_t = ddpm.schedule.inflate_batch_array(ddpm.schedule.gamma_module(t_noise), batch['h'])

torch.manual_seed(1)
z_t, eps_xh = ddpm.noised_representation(batch_norm['h'], batch_norm['mask'], gamma_t)

print('z_t   ', tuple(z_t.shape), z_t.dtype)
print('eps_xh', tuple(eps_xh.shape), eps_xh.dtype)
print('max |z_t|   ', float(z_t.abs().max()))
print('max |eps_xh|', float(eps_xh.abs().max()))

assert z_t.shape == batch['h'].shape
assert eps_xh.shape == batch['h'].shape
assert z_t.dtype == batch['h'].dtype
assert eps_xh.dtype == batch['h'].dtype
assert torch.isfinite(z_t).all()
assert torch.isfinite(eps_xh).all()


z_t    (6, 122) torch.float64
eps_xh (6, 122) torch.float64
max |z_t|    8.454450752271875
max |eps_xh| 1.789062261581421


## 5.2 Diffusion Noising Input / Output Contract

`Diffusion.noised_representation` 输入干净的 `h` 和 `gamma_t`，输出加噪后的 `z_t` 以及监督信号真实噪声 `eps_xh`。


In [15]:
print_tensor_contract('Diffusion.noised_representation inputs', {
    'batch_norm_h': batch_norm['h'],
    'mask': batch_norm['mask'],
    't_noise': t_noise,
    'gamma_t': gamma_t,
})
print_tensor_contract('Diffusion.noised_representation outputs', {
    'z_t': z_t,
    'eps_xh': eps_xh,
})



[Diffusion.noised_representation inputs]
batch_norm_h     shape=(6, 122)           dtype=torch.float64  device=cpu
mask             shape=(6,)               dtype=torch.int64    device=cpu
t_noise          shape=(1, 1)             dtype=torch.float64  device=cpu
gamma_t          shape=(1, 1)             dtype=torch.float64  device=cpu

[Diffusion.noised_representation outputs]
z_t              shape=(6, 122)           dtype=torch.float64  device=cpu
eps_xh           shape=(6, 122)           dtype=torch.float64  device=cpu


## 6. One Diffusion Loss Forward

In [16]:
ddpm.train()
torch.manual_seed(2)
loss_terms = ddpm(batch, conditions=None)

loss_keys = ['error_t', 'loss_0_x', 'loss_0_cat', 'loss_0_charge', 'kl_prior', 'delta_log_px', 'SNR_weight']
for key in loss_keys:
    value = loss_terms[key]
    print(f'{key:14s}', tuple(value.shape), value.dtype, 'mean =', float(value.detach().mean()))
    assert torch.isfinite(value).all(), key

print('t_int:', loss_terms['t_int'].detach().cpu().tolist())
print('net_eps_xh', tuple(loss_terms['net_eps_xh'].shape), loss_terms['net_eps_xh'].dtype)
print('eps_xh    ', tuple(loss_terms['eps_xh'].shape), loss_terms['eps_xh'].dtype)

assert loss_terms['net_eps_xh'].shape == batch['h'].shape
assert loss_terms['eps_xh'].shape == batch['h'].shape


error_t        (1,) torch.float64 mean = 16.664394411337945
loss_0_x       (1,) torch.float64 mean = 0.0
loss_0_cat     (1,) torch.float64 mean = 0.0
loss_0_charge  (1,) torch.float64 mean = 0.0
kl_prior       (1,) torch.float64 mean = 0.0
delta_log_px   (1,) torch.float64 mean = 0.0
SNR_weight     (1,) torch.float64 mean = -0.7308157096668222
t_int: [3.0]
net_eps_xh (6, 122) torch.float64
eps_xh     (6, 122) torch.float64


## 6.1 Diffusion Forward Output Contract

`Diffusion.forward` 返回 loss 需要的所有中间量，其中 `net_eps_xh` 是 Denoiser 预测噪声，`eps_xh` 是真实采样噪声。


In [17]:
print_tensor_contract('Diffusion.forward selected outputs', {
    'net_eps_xh': loss_terms['net_eps_xh'],
    'eps_xh': loss_terms['eps_xh'],
    't_int': loss_terms['t_int'],
    'error_t': loss_terms['error_t'],
    'SNR_weight': loss_terms['SNR_weight'],
})



[Diffusion.forward selected outputs]
net_eps_xh       shape=(6, 122)           dtype=torch.float64  device=cpu
eps_xh           shape=(6, 122)           dtype=torch.float64  device=cpu
t_int            shape=(1,)               dtype=torch.float64  device=cpu
error_t          shape=(1,)               dtype=torch.float64  device=cpu
SNR_weight       shape=(1,)               dtype=torch.float64  device=cpu


## 7. One Backward Pass

In [18]:
for param in ddpm.parameters():
    param.grad = None

torch.manual_seed(3)
loss_terms = ddpm(batch, conditions=None)
total_loss = (
    loss_terms['error_t']
    + loss_terms['loss_0_x']
    + loss_terms['loss_0_cat']
    + loss_terms['loss_0_charge']
    + loss_terms['kl_prior']
).mean()
total_loss.backward()

grad_sq_sum = 0.0
grad_param_count = 0
for param in ddpm.parameters():
    if param.grad is not None:
        grad_sq_sum += float(param.grad.detach().pow(2).sum())
        grad_param_count += 1
grad_norm = grad_sq_sum ** 0.5

print('total_loss =', float(total_loss.detach()))
print('grad_param_count =', grad_param_count)
print('grad_norm =', grad_norm)

assert torch.isfinite(total_loss)
assert grad_param_count > 0
assert grad_norm > 0

print('H2O Denoiser + Diffusion debug passed.')


total_loss = 10.070222072794234
grad_param_count = 74
grad_norm = 65.0480783242224
H2O Denoiser + Diffusion debug passed.
